In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


In [ ]:
#Load Dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
data = pd.read_csv(url, sep="\t", names=["label", "message"])

data.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#Label Encoding + Split
data["label"] = data["label"].map({"ham": 0, "spam": 1})

X = data["message"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
#TF-IDF Vectorization (CORE NLP STEP)
vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [ ]:
#NLP + Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_preds = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_preds))


Naive Bayes Accuracy: 0.9739910313901345


In [ ]:
#NLP + Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

lr_preds = lr_model.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_preds))


Logistic Regression Accuracy: 0.9721973094170404


In [ ]:
#NLP + SVM (BEST)
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, svm_preds))


SVM Accuracy: 0.9829596412556054


In [ ]:
#Final Evaluation
print("\n📊 Classification Report (SVM):")
print(classification_report(y_test, svm_preds))



📊 Classification Report (SVM):
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.99      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
#User Input Spam Checker
def predict_spam(message):
    message_tfidf = vectorizer.transform([message])
    prediction = svm_model.predict(message_tfidf)[0]

    if prediction == 1:
        return "🚨 Spam Message"
    else:
        return "✅ Not Spam (Ham)"

# Test examples
print(predict_spam("Congratulations! You won a free lottery. Call now"))
print(predict_spam("Hey, are we meeting today?"))


🚨 Spam Message
✅ Not Spam (Ham)
